In [1]:
import pandas as pd
import spacy

In [2]:
data = pd.read_csv('spam.csv')

In [3]:
data.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Category  5572 non-null   object
 1   Message   5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [5]:
data['Category'].value_counts()

Category
ham     4825
spam     747
Name: count, dtype: int64

In [6]:
nlp = spacy.load("en_core_web_sm")

In [7]:
stop_words = nlp.Defaults.stop_words
len(stop_words)
stop_words.remove('do')
nlp.vocab['do'].is_stop=False
# stop_words.add('offer')
# nlp.vocab['offer'].is_stop=True

In [8]:
corpus=[]
for i in range(0,data.shape[0]):
    words=[]
    doc = nlp(data['Message'][i])
    for token in doc:
        if(token.lemma_ not in stop_words):
            words.append(token.lemma_)
    new_text = " ".join(words)
    corpus.append(new_text)

In [9]:
new_text

'rofl . true'

In [10]:
data["Message"][0]

'Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...'

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
tf = TfidfVectorizer()
output = tf.fit_transform(corpus)
output

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 41964 stored elements and shape (5572, 7610)>

In [12]:
output= output.toarray()

In [13]:
#unique words from corpus which will be columns in input matrix
# tf.get_feature_names_out()
# We are printing 50 words
print(tf.get_feature_names_out()[2000:2050])

['completely' 'complexity' 'compliment' 'complimentary' 'comprehensive'
 'compromise' 'compulsory' 'computational' 'computer' 'computerless'
 'comuk' 'con' 'conacte' 'concentrate' 'concentration' 'concern'
 'concerned' 'concert' 'conclusion' 'condition' 'conduct' 'conected'
 'conference' 'confidence' 'configure' 'confirm' 'confirmd' 'conform'
 'confuse' 'confused' 'congrat' 'congrats' 'congratulation' 'connect'
 'connection' 'consensus' 'consent' 'conserve' 'consider' 'consistently'
 'console' 'constant' 'constantly' 'contact' 'contain' 'content'
 'contention' 'continent' 'continue' 'continued']


In [14]:
# print(data["Message"])
# rows where complimentry word is not 0
temp = pd.DataFrame(output)
temp[temp[2003]!=0]

,0,1,2,3,4,5,6,7,8,9,...,7600,7601,7602,7603,7604,7605,7606,7607,7608,7609
67,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
844,0.0,0.207236,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
935,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
939,0.0,0.215372,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1888,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1985,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2730,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3382,0.0,0.251535,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4183,0.0,0.210187,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4450,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [15]:
#one of the sentence where there is complimentory word
print(data['Message'][844])

Urgent! call 09066350750 from your landline. Your complimentary 4* Ibiza Holiday or 10,000 cash await collection SAE T&Cs PO BOX 434 SK3 8WP 150 ppm 18+


In [16]:
from sklearn.model_selection import train_test_split
X_train,X_test , y_train, y_test  = train_test_split(output, data['Category'],
                                                     test_size=0.3,random_state=45)

In [17]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
model = MultinomialNB()
model.fit(X_train,y_train)
print("Training Acc:", accuracy_score(model.predict(X_train),y_train))
print("Test Acc:", accuracy_score(model.predict(X_test),y_test))
      

Training Acc: 0.978974358974359
Test Acc: 0.9623205741626795


In [18]:
msg = "I am going to be late for office tomorrow. Will complete my task accourdingly"
doc = nlp(msg)
for token in doc:
   if(token.lemma_ not in stop_words):
            words.append(token.lemma_)
new_text = " ".join(words)
input_data = tf.transform([new_text])
model.predict(input_data)

array(['ham'], dtype='<U4')